In [1]:
# utils.py

import os
from datetime import datetime, date
from dotenv import load_dotenv

load_dotenv()

GOOGLE_SHEETS_ID = os.getenv("GOOGLE_SHEETS_ID")

def get_today_str_iso():
    """YYYY-MM-DD, para registro de data completo (não usado na célula)"""
    return date.today().isoformat()

def get_today_day():
    """Retorna apenas o dia do mês (1 a 31)."""
    return datetime.today().day

def get_columns_for_today():
    """
    Calcula colunas da estrutura única de planilha.

    Estrutura (linha 2):
    A: mês JAN,   A2: "Data"  B2:"Entrada" C2:"Saída" D2:"Diário" E2:"Saldo" F2:""
    G: mês FEV,   G2: "Data"  H2:"Entrada" I2:"Saída" J2:"Diário" K2:"Saldo" L2:""
    M: mês MAR, ...

    Cada bloco de mês tem 6 colunas (Data, Entrada, Saída, Diário, Saldo, Vazia).
    Índices 1-based (A=1, B=2, ...).
    """
    # offset em colunas por mês (0, 6, 12, 18, ...)
    meses_offset = {
        1: 0,   # JAN
        2: 6,   # FEV
        3: 12,  # MAR
        4: 18,  # ABR
        5: 24,  # MAI
        6: 30,  # JUN
        7: 36,  # JUL
        8: 42,  # AGO
        9: 48,  # SET
        10:54,  # OUT
        11:60,  # NOV
        12:66   # DEZ
    }
    hoje = datetime.now()
    mes_offset = meses_offset[hoje.month]
    dia = hoje.day
    row = 2 + dia  # linha 3 para dia 1, linha 4 para dia 2, ...

    data_col = 1 + mes_offset
    entrada_col = 2 + mes_offset
    saida_col = 3 + mes_offset
    diario_col = 4 + mes_offset
    saldo_col = 5 + mes_offset

    return {
        "row": row,
        "data_col": data_col,
        "entrada_col": entrada_col,
        "saida_col": saida_col,
        "diario_col": diario_col,
        "saldo_col": saldo_col
    }

def get_sheet_name():
    """Nome da aba é o ano atual, ex: "2024"."""
    return str(datetime.now().year)

In [2]:
print(f"Today's date (ISO): {get_today_str_iso()}")
print(f"Today's day: {get_today_day()}")
print(f"Today's columns: {get_columns_for_today()}")

Today's date (ISO): 2026-02-19
Today's day: 19
Today's columns: {'row': 21, 'data_col': 7, 'entrada_col': 8, 'saida_col': 9, 'diario_col': 10, 'saldo_col': 11}


In [6]:
# gemini_parser.py

from google import genai
import json
import os
from dotenv import load_dotenv
# from utils import get_today_str_iso

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

MODEL_NAME = "gemini-2.5-flash-lite"

def parse_audio_expense(audio_path: str):
    """
    Transcreve o áudio e retorna um dict:
    {
      "tipo": "receita" | "despesa_fixa" | "despesa_diaria",
      "valor": float,
      "categoria": str,
      "data": "YYYY-MM-DD",
      "descricao": str
    }
    """
    # instruções claras para o modelo
    prompt = f"""
    Você é um assistente financeiro que classifica gastos e receitas.

    REGRAS PARA TIPOS:
    - "receita": dinheiro que entra (salário, reembolso, rendimentos, etc.).
    - "despesa_fixa": contas mensais ou recorrentes (energia, água, gás, condomínio, aluguel, wifi, telefone, cartão de crédito, seguro, etc.).
    - "despesa_diaria": gastos do dia a dia (mercado, restaurante, lanchonete, combustível, transporte, farmácia, bar, lazer, etc.).

    Exemplos:
    - "Gastei 50 reais de energia" -> tipo = "despesa_fixa"
    - "Paguei o condomínio hoje" -> tipo = "despesa_fixa"
    - "Gastei 20 reais no mercado" -> tipo = "despesa_diaria"
    - "Comprei um lanche de 15" -> tipo = "despesa_diaria"
    - "Recebi 500 de salário" -> tipo = "receita"

    CAMPO DATA:
    - Se o áudio falar "hoje", "agora" ou não falar data, use "{get_today_str_iso()}".
    - Se mencionar explicitamente uma data (ex: "dia 10", "10 de fevereiro"), converta para o formato "YYYY-MM-DD" correto.

    FORMATO DE RESPOSTA:
    Responda APENAS com um JSON válido, sem texto extra, no formato:

    {{
      "tipo": "receita" | "despesa_fixa" | "despesa_diaria",
      "valor": 13.8,  // use ponto como separador decimal
      "categoria": "mercado",
      "data": "YYYY-MM-DD",
      "descricao": "Gasto no mercado"
    }}
    """
    audio_file = client.files.upload(audio_path)
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=[prompt, audio_file],
        config={
            "response_mime_type": "application/json"
        }
    )
    try:
        text = response.text.strip()
        if text.startswith("```json"):
            text = text[7:-3].strip()  # remove ```json ... ```
        elif text.startswith("```"):
            text = text[3:-3].strip()  # remove ``` ... ```
        
        data = json.loads(text)

        # normalização/validações
        data["valor"] = float(data["valor"])
        
        if not data.get("tipo"):
            # fallback: tudo que não for receita vira despesa_diaria
            data["tipo"] = "despesa_diaria"

        if data["tipo"] not in ["receita", "despesa_fixa", "despesa_diaria"]:
            data["tipo"] = "despesa_diaria"

        if not data.get("data"):
            data["data"] = get_today_str_iso()
        
        if not data.get("descricao"):
            data["descricao"] = f"{data['tipo']} de {data['valor']}"
        
        if not data.get("categoria"):
            data["categoria"] = "outros"

        return data
    
    except json.JSONDecodeError:
        raise ValueError(f"Resposta do modelo não é um JSON válido: {response.text}")
    finally:
        client.files.delete(audio_file.name)

In [7]:
def parse_text_expense(texto: str):
    prompt = f"""
    Você é um assistente financeiro que classifica gastos e receitas.

    REGRAS PARA TIPOS:
    - "receita": dinheiro que entra (salário, reembolso, rendimentos, etc.).
    - "despesa_fixa": contas mensais ou recorrentes (energia, água, gás, condomínio, aluguel, wifi, telefone, cartão de crédito, seguro, etc.).
    - "despesa_diaria": gastos do dia a dia (mercado, restaurante, lanchonete, combustível, transporte, farmácia, bar, lazer, etc.).

    Exemplos:
    - "Gastei 50 reais de energia" -> tipo = "despesa_fixa"
    - "Paguei o condomínio hoje" -> tipo = "despesa_fixa"
    - "Gastei 20 reais no mercado" -> tipo = "despesa_diaria"
    - "Comprei um lanche de 15" -> tipo = "despesa_diaria"
    - "Recebi 500 de salário" -> tipo = "receita"

    CAMPO DATA:
    - Se o áudio falar "hoje", "agora" ou não falar data, use "{get_today_str_iso()}".
    - Se mencionar explicitamente uma data (ex: "dia 10", "10 de fevereiro"), converta para o formato "YYYY-MM-DD" correto.

    FORMATO DE RESPOSTA:
    Responda APENAS com um JSON válido, sem texto extra, no formato:

    {{
      "tipo": "receita" | "despesa_fixa" | "despesa_diaria",
      "valor": 13.8,  // use ponto como separador decimal
      "categoria": "mercado",
      "data": "YYYY-MM-DD",
      "descricao": "Gasto no mercado"
    }}

    TEXTO: {texto}
    """

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=[prompt],
        config={"response_mime_type": "application/json"}
    )

    try:
        text = response.text.strip()
        if text.startswith("```json"):
            text = text[7:-3].strip()  # remove ```json ... ```
        elif text.startswith("```"):
            text = text[3:-3].strip()  # remove ``` ... ```
        
        data = json.loads(text)

        # normalização/validações
        data["valor"] = float(data["valor"])
        
        if not data.get("tipo"):
            # fallback: tudo que não for receita vira despesa_diaria
            data["tipo"] = "despesa_diaria"

        if data["tipo"] not in ["receita", "despesa_fixa", "despesa_diaria"]:
            data["tipo"] = "despesa_diaria"

        if not data.get("data"):
            data["data"] = get_today_str_iso()
        
        if not data.get("descricao"):
            data["descricao"] = f"{data['tipo']} de {data['valor']}"
        
        if not data.get("categoria"):
            data["categoria"] = "outros"

        return data
    
    except json.JSONDecodeError:
        raise ValueError(f"Resposta do modelo não é um JSON válido: {response.text}")

result = parse_text_expense("Gastei 20 reais no mercado, hoje")
print(result)

{'tipo': 'despesa_diaria', 'valor': 20.0, 'categoria': 'mercado', 'data': '2026-02-19', 'descricao': 'Gasto no mercado'}


In [16]:
# sheets_writer.py

import gspread
from google.oauth2.service_account import Credentials
from dotenv import load_dotenv
# from utils import get_columns_for_today, get_sheet_name, GOOGLE_SHEETS_ID

load_dotenv()

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

def get_sheets_client():
    """
    Cria cliente autenticado para Google Sheets usando credenciais de conta de serviço.
    """
    creds = Credentials.from_service_account_file(
        "credentials.json",
        scopes=SCOPES
    )
    gc = gspread.authorize(creds)
    return gc

def _get_cell_float(value: str) -> float:
    """
    Converte valor da célula (string) para float, tratando vazio e removendo "R$" e vírgulas. Ex: "R$ 1.234,56" -> 1234.56
    """
    if value is None or value == "":
        return 0.0
    
    cleaned = value.replace("R$", "").replace(".", "").replace(",", ".").strip()
    try:
        return float(cleaned)
    except ValueError:
        return 0.0

def append_expense_to_sheet(parsed_data):
    """
    Atualiza a linha do dia atual:
    - receita -> soma em "Entrada"
    - despesa_fixa -> soma em "Saída"
    - despesa_diaria -> soma em "Diário"

    Se já houver valor naquele dia e valor for diferente de 33,36.
    """
    gc = get_sheets_client()
    try:
        sh = gc.open_by_key(GOOGLE_SHEETS_ID)
        ws = sh.worksheet(get_sheet_name())
    except gspread.WorksheetNotFound:
        raise ValueError(f"Aba '{get_sheet_name()}' não encontrada na planilha.")
    
    pos = get_columns_for_today()
    row = pos["row"]

    # Garante que a coluna Data do dia foi preenchida (caso esteja vazia)
    data_cell = ws.cell(row, pos["data_col"]).value
    if not data_cell:
        ws.update_cell(row, pos["data_col"], row - 2)  # preenche o dia do mês (linha 3 = dia 1, linha 4 = dia 2, ...)

    valor = float(parsed_data["valor"])
    if parsed_data["tipo"] == "receita":
        target_col = pos["entrada_col"]
    elif parsed_data["tipo"] == "despesa_fixa":
        target_col = pos["saida_col"]
    else:  # despesa_diaria
        target_col = pos["diario_col"] 

    # lê o valor atual da célula, trata vazio como 0, e soma o novo valor se for diferente de 33,36
    atual_str = ws.cell(row, target_col).value
    atual = _get_cell_float(atual_str)
    print(f"Valor atual na célula (linha {row}, coluna {target_col}): '{atual_str}' -> {atual:.2f}")
    if valor != 33.36:
        novo_valor = atual + valor
    else:
        novo_valor = valor  # caso especial para 33,36, não soma, apenas registra o valor
    ws.update_cell(row, target_col, novo_valor)

    return (
        f"Dia {row - 2}: tipo={parsed_data['tipo']}"
        f"↑ {valor:.2f} (total agora {novo_valor:.2f})"
    )

In [18]:
test_case = {
    "tipo": "despesa_diaria",
    "valor": 35.0,
    "categoria": "lorcana",
    "data": get_today_str_iso(),
    "descricao": "Compra de cards do jogo lorcana"
}
result = append_expense_to_sheet(test_case)
print(result)

Valor atual na célula (linha 21, coluna 10): 'R$ 33,36' -> 33.36
Dia 19: tipo=despesa_diaria↑ 35.00 (total agora 68.36)
